# 005 Rule-Based Triage Safety Baseline

Inspect the first deterministic routing and safety baseline on dev fixtures only. This notebook is a review surface; it should not train, index, load models, or generate troubleshooting answers.

## Stage Contract

- Use only question-only eval fixtures from `004`.
- Keep holdout files untouched during dev tuning.
- Compare `text_only` and `metadata_assisted` profiles honestly.
- Treat Stack Exchange tags as source metadata, not normal user-ticket text.
- Record safety false negatives for manual review before claiming safety performance.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT = ROOT / 'data' / 'eval' / 'rule_based_triage_safety_baseline'
SCRIPT = ROOT / 'scripts' / 'evaluate_rule_based_triage_safety.py'

SUMMARY = OUT / 'baseline_summary_dev_only.json'

## Regenerate Dev Baseline

Leave `RUN_EVAL` false unless you are intentionally refreshing the fast dev-only baseline. Do not pass `--include-holdout` during dev tuning.

In [ ]:
RUN_EVAL = False

if RUN_EVAL:
    result = subprocess.run(
        [sys.executable, str(SCRIPT), '--summary'],
        cwd=ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    print(result.stdout)

assert SUMMARY.exists(), f'Missing baseline summary: {SUMMARY}'

## Summary

In [ ]:
display(Markdown((OUT / 'baseline_summary_dev_only.md').read_text(encoding='utf-8')))
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
summary['input_counts']

## Routing Metrics

In [ ]:
routing_metrics = pd.read_csv(OUT / 'routing_metrics_dev_only.csv')
display(routing_metrics.pivot(index='domain', columns='profile', values='f1'))
display(routing_metrics.sort_values(['profile', 'f1']))

## Safety Metrics

In [ ]:
safety_metrics = pd.read_csv(OUT / 'safety_metrics_dev_only.csv')
display(safety_metrics)

## False Negative Samples

In [ ]:
rows = []
for profile in summary['profiles']:
    for expected_behavior, samples in profile['safety_false_negative_samples'].items():
        for sample in samples[:10]:
            rows.append({
                'profile': profile['profile'],
                'expected_behavior': expected_behavior,
                'case_id': sample['case_id'],
                'title': sample['title'],
                'predicted_behavior': sample['predicted_behavior'],
                'predicted_primary_domain': sample['predicted_primary_domain'],
                'question_tags': ', '.join(sample.get('question_tags', [])),
            })

pd.DataFrame(rows)

## Prediction Samples

In [ ]:
predictions = pd.read_json(OUT / 'routing_predictions_dev_only_text_only.jsonl', lines=True)
predictions[['case_id', 'title', 'expected_primary_domain', 'predicted_primary_domain', 'predicted_domains']].head(20)

## Exit Criteria

- Baseline report exists for dev fixtures only.
- No holdout files were read during normal notebook inspection.
- No models, embeddings, FAISS indexes, training data, or answer-generation records were produced.
- Per-domain routing metrics and safety false-negative counts are visible.
- Next step is manual review of high-impact safety misses and source-gate conservatism before any holdout run.